# 03a — Dataset Analysis

**Purpose:** Analyze the Chest X-Ray (Pneumonia) dataset to understand class distributions, image properties, and potential biases.

| Input | Output |
|---|---|
| Raw dataset (via kagglehub) | Visualizations → `results/figures/dataset/` |

**Runtime:** ~5 minutes (no GPU required)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import kagglehub
import json
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

DATA_PATH = Path(kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia'))
RESULTS_DIR = Path('results')
FIG_DIR = RESULTS_DIR / 'figures' / 'dataset'
for d in [FIG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_COLORS = ['#1B4F8A', '#C0392B']

## 1. Dataset Splits Overview

In [ ]:
def find_dataset_root(base: Path) -> Path:
    for p in sorted(base.rglob('train')):
        if p.is_dir() and '__MACOSX' not in p.parts and (p.parent / 'test').is_dir():
            return p.parent
    raise FileNotFoundError(f'Could not find train/test under {base}')

DATA_PATH = find_dataset_root(DATA_PATH)

def collect_split(data_path: Path, split: str):
    paths, labels = [], []
    for label, cls in [(0, 'NORMAL'), (1, 'PNEUMONIA')]:
        for p in sorted((data_path / split / cls).glob('*.jpeg')):
            paths.append(p)
            labels.append(label)
    return paths, np.array(labels, dtype=np.int64)

# Collect all splits
train_paths, train_labels = collect_split(DATA_PATH, 'train')
val_paths,   val_labels   = collect_split(DATA_PATH, 'val')
test_paths,  test_labels  = collect_split(DATA_PATH, 'test')

# Compute statistics
def summarize(name, paths, labels):
    n_total = len(paths)
    n_normal = (labels == 0).sum()
    n_pneumo = (labels == 1).sum()
    pct_pneumo = 100 * n_pneumo / n_total
    return {'split': name, 'total': n_total, 'normal': n_normal, 'pneumonia': n_pneumo, 'pct': pct_pneumo}

summary = [
    summarize('train', train_paths, train_labels),
    summarize('val', val_paths, val_labels),
    summarize('test', test_paths, test_labels)
]

df_summary = pd.DataFrame(summary)
print("=== Dataset Split Summary ===")
print(df_summary.to_string(index=False))

## 2. Visualize Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

splits = ['train', 'val', 'test']

# Left: Stacked bar
bottoms = [100 - p for p in df_summary['pct']]
axes[0].bar(splits, bottoms, label='Normal', color=CLASS_COLORS[0])
axes[0].bar(splits, df_summary['pct'], bottom=bottoms, label='Pneumonia', color=CLASS_COLORS[1])
axes[0].set_ylabel('Percentage (%)')
axes[0].set_title('Class distribution by split')
axes[0].legend()
axes[0].set_ylim(0, 100)
axes[0].grid(True, axis='y', alpha=0.3)

# Right: Pneumonia % line
axes[1].plot(splits, df_summary['pct'], 'o-', color=CLASS_COLORS[1], lw=2, markersize=10)
axes[1].set_ylabel('Pneumonia %')
axes[1].set_title('Pneumonia prevalence')
axes[1].set_ylim(0, 100)
axes[1].axhline(50, color='gray', linestyle='--', alpha=0.5, label='50%')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

for i, pct in enumerate(df_summary['pct']):
    axes[1].annotate(f'{pct:.1f}%', (i, pct+2), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(FIG_DIR / 'class_distribution.pdf', bbox_inches='tight')
plt.show()
print(f'Saved → {FIG_DIR}/class_distribution.pdf')

## 3. Image Properties

In [ ]:
np.random.seed(42)
sample_size = 200
all_paths = train_paths + test_paths
sampled = np.random.choice(all_paths, min(sample_size, len(all_paths)), replace=False)

widths, heights, modes = [], [], []
for p in sampled:
    img = Image.open(p)
    widths.append(img.width)
    heights.append(img.height)
    modes.append(img.mode)

print(f'Analyzed {len(sampled)} images:')
print(f'  Width:  min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.0f}')
print(f'  Height: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.0f}')
print(f'  Modes:  {dict(pd.Series(modes).value_counts())}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Size scatter
axes[0].scatter(widths, heights, alpha=0.5, s=20)
axes[0].set_xlabel('Width')
axes[0].set_ylabel('Height')
axes[0].set_title('Image sizes')
axes[0].axline(224, slope=1, color='red', linestyle='--', alpha=0.7, label='224×224')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Area histogram
areas = [w*h for w, h in zip(widths, heights)]
axes[1].hist(areas, bins=30, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].set_xlabel('Area')
axes[1].set_ylabel('Count')
axes[1].set_title('Image area distribution')
axes[1].axvline(np.median(areas), color='red', linestyle='--', label=f'Median: {np.median(areas):.0f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'image_sizes.pdf', bbox_inches='tight')
plt.show()
print(f'saved → {FIG_DIR}/image_sizes.pdf')

## 4. Key Insights

In [ ]:
train_pct = df_summary.loc[df_summary['split'] == 'train', 'pct'].values[0]
test_pct = df_summary.loc[df_summary['split'] == 'test', 'pct'].values[0]
shift = train_pct - test_pct

print('=== Key Dataset Insights ===\n')
print(f'1. CLASS IMBALANCE:')
print(f'   Train: {train_pct:.1f}% pneumonia')
print(f'   Test:  {test_pct:.1f}% pneumonia')
print(f'   Gap:   {shift:.1f} pp\n')

print('2. VAL SET TOO SMALL:')
print(f'   Original val: {len(val_paths)} images\n')

print('3. IMAGE SIZE VARIATION:')
print(f'   Range: {min(widths)}-{max(widths)} px\n')

# Save insights
insights = {
    'train_pneumonia_pct': float(train_pct),
    'test_pneumonia_pct': float(test_pct),
    'class_shift_pp': float(shift),
    'val_size': len(val_paths),
    'train_size': len(train_paths),
    'test_size': len(test_paths),
    'image_width_range': [int(min(widths)), int(max(widths))],
    'image_height_range': [int(min(heights)), int(max(heights))]
}

with open(RESULTS_DIR / 'dataset_insights.json', 'w') as f:
    json.dump(insights, f, indent=2)
print(f'saved → {RESULTS_DIR}/dataset_insights.json')

## 5. Summary

| Check | Status |
|:---|:---:|
| Dataset splits verified | ✅ |
| Class distribution visualized | ✅ |
| Image properties analyzed | ✅ |
| Insights documented | ✅ |

**Next:** Run `02_preprocessing.ipynb` before analysis notebooks.